# Deep Dive: Build the Cat Health Vector RAG Pipeline From Scratch

The main Session 1 notebook builds a dense vector retrieval application with **LangChain**, **OpenAI embeddings**, and **Qdrant**. This optional reference notebook rebuilds the same pipeline with the abstractions exposed.

We will still use `pypdf` to extract text from the PDF. After extraction, everything is implemented here with Python standard-library code: raw HTTP requests, a small provider adapter, document objects, character chunking, embeddings, cosine similarity, brute-force in-memory vector search, prompt augmentation, and generation.

The goal is not to create a production-ready SDK. The goal is to understand what production libraries do for us.

```text
question -> embed query -> compare vectors -> retrieve chunks -> augment prompt -> generate answer
```

## Table of Contents

- 1. Environment Setup
- 2. Raw HTTP Transport
- 3. Provider-Neutral Model Primitives
- 4. Embedding Similarity Primer
- 5. Documents - Loading the Cat Health Guideline PDF
- 6. Chunking the Documents
- 7. Build an In-Memory Vector Store
- 8. Retrieval with Scores
- 9. Retrieval Augmented Generation
- Production Library Responsibilities

## 1. Environment Setup

From the `01_Dense_Vector_Retrieval` folder, install dependencies with uv:

```bash
uv sync
```

Then open this notebook in Cursor or VS Code and select the Python/Jupyter environment created by uv.

### Imports

The only third-party import in this notebook is `pypdf`. It handles PDF extraction so we can focus on the retrieval abstractions.

In [ ]:
from dataclasses import dataclass
from getpass import getpass
from math import sqrt
from pathlib import Path
from typing import Any, Protocol
from urllib.error import HTTPError, URLError
from urllib.request import Request, urlopen
import json
import os

from pypdf import PdfReader

### OpenAI API Key and Model Defaults

The embedding and language-model adapters both call OpenAI directly. If `OPENAI_API_KEY` is not already set in your environment, this cell will ask for it securely.

The defaults match the main Session 1 notebook. You can override them with environment variables before running the notebook.

In [ ]:
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("OpenAI API Key: ")

openai_base_url = os.environ.get("OPENAI_BASE_URL", "https://api.openai.com/v1").rstrip("/")
embedding_model_id = os.environ.get("AIM_EMBEDDING_MODEL", "text-embedding-3-small")
chat_model_id = os.environ.get("AIM_CHAT_MODEL", "gpt-5.4-mini")

print(f"Embedding model: {embedding_model_id}")
print(f"Chat model: {chat_model_id}")

## 2. Raw HTTP Transport

Provider SDKs handle request construction, authentication, JSON serialization, JSON parsing, and API errors. Our `_post_json()` helper makes that boundary visible.

It intentionally does **not** implement retries, exponential backoff, streaming, telemetry, or concurrent requests.

In [ ]:
def _post_json(
    *,
    path: str,
    payload: dict[str, Any],
    api_key: str,
    base_url: str,
    timeout: int = 60,
) -> dict[str, Any]:
    """POST a JSON payload and return the decoded JSON response."""
    if not api_key:
        raise ValueError("An OpenAI API key is required.")

    request = Request(
        url=f"{base_url}{path}",
        data=json.dumps(payload).encode("utf-8"),
        headers={
            "Authorization": f"Bearer {api_key}",
            "Content-Type": "application/json",
        },
        method="POST",
    )

    try:
        with urlopen(request, timeout=timeout) as response:
            response_body = response.read().decode("utf-8")
    except HTTPError as exc:
        error_body = exc.read().decode("utf-8", errors="replace")
        try:
            error_payload = json.loads(error_body)
            error_message = error_payload.get("error", {}).get("message", error_body)
        except json.JSONDecodeError:
            error_message = error_body
        raise RuntimeError(
            f"OpenAI API request failed with HTTP {exc.code}: {error_message}"
        ) from exc
    except URLError as exc:
        raise RuntimeError(f"OpenAI API request could not connect: {exc.reason}") from exc

    try:
        return json.loads(response_body)
    except json.JSONDecodeError as exc:
        raise RuntimeError("OpenAI API returned a response that was not valid JSON.") from exc

## 3. Provider-Neutral Model Primitives

The application should not need to know which HTTP endpoint an embedding model or language model uses. We will put provider-specific behavior behind two small protocols:

- `EmbeddingModel.do_embed()` turns many strings into many vectors.
- `LanguageModel.do_generate()` turns instructions and a prompt into generated text.

The result dataclasses surface token usage and the raw provider response. Production SDKs often expose similar metadata while also normalizing differences between providers.

In [ ]:
@dataclass
class EmbedResult:
    value: str
    embedding: list[float]
    usage: dict[str, Any]
    raw_response: dict[str, Any]


@dataclass
class EmbedManyResult:
    values: list[str]
    embeddings: list[list[float]]
    usage: dict[str, Any]
    raw_response: dict[str, Any]


@dataclass
class GenerateTextResult:
    text: str
    usage: dict[str, Any]
    raw_response: dict[str, Any]


class EmbeddingModel(Protocol):
    def do_embed(self, values: list[str]) -> EmbedManyResult:
        ...


class LanguageModel(Protocol):
    def do_generate(self, *, instructions: str, prompt: str) -> GenerateTextResult:
        ...

### OpenAI Provider Adapter

The OpenAI embeddings endpoint accepts one string or an array of strings. The API returns an `index` for each vector, so the adapter sorts by that index before handing vectors back to application code.

The Responses API returns structured output items. The helper below flattens `output_text` parts into the text our RAG application needs.

In [ ]:
def _extract_output_text(response: dict[str, Any]) -> str:
    """Flatten output_text parts from a raw Responses API payload."""
    text_parts = []

    for output_item in response.get("output", []):
        for content_part in output_item.get("content", []):
            if content_part.get("type") == "output_text":
                text_parts.append(content_part.get("text", ""))

    return "".join(text_parts)


@dataclass(frozen=True)
class OpenAIEmbeddingModel:
    model_id: str
    api_key: str
    base_url: str

    def do_embed(self, values: list[str]) -> EmbedManyResult:
        if not values:
            raise ValueError("At least one value is required for embedding.")
        if any(not value.strip() for value in values):
            raise ValueError("Embedding values must not be empty strings.")

        raw_response = _post_json(
            path="/embeddings",
            payload={
                "model": self.model_id,
                "input": values,
                "encoding_format": "float",
            },
            api_key=self.api_key,
            base_url=self.base_url,
        )

        ordered_data = sorted(raw_response.get("data", []), key=lambda item: item["index"])
        embeddings = [item["embedding"] for item in ordered_data]

        if len(embeddings) != len(values):
            raise RuntimeError("OpenAI API returned an unexpected number of embeddings.")

        return EmbedManyResult(
            values=list(values),
            embeddings=embeddings,
            usage=raw_response.get("usage", {}),
            raw_response=raw_response,
        )


@dataclass(frozen=True)
class OpenAILanguageModel:
    model_id: str
    api_key: str
    base_url: str

    def do_generate(self, *, instructions: str, prompt: str) -> GenerateTextResult:
        raw_response = _post_json(
            path="/responses",
            payload={
                "model": self.model_id,
                "instructions": instructions,
                "input": prompt,
            },
            api_key=self.api_key,
            base_url=self.base_url,
        )
        text = _extract_output_text(raw_response)

        if not text:
            raise RuntimeError("OpenAI Responses API returned no output_text content.")

        return GenerateTextResult(
            text=text,
            usage=raw_response.get("usage", {}),
            raw_response=raw_response,
        )


@dataclass(frozen=True)
class OpenAIProvider:
    api_key: str
    base_url: str = "https://api.openai.com/v1"

    def embedding_model(self, model_id: str) -> OpenAIEmbeddingModel:
        return OpenAIEmbeddingModel(model_id=model_id, api_key=self.api_key, base_url=self.base_url)

    def language_model(self, model_id: str) -> OpenAILanguageModel:
        return OpenAILanguageModel(model_id=model_id, api_key=self.api_key, base_url=self.base_url)

### Application-Facing Helpers

These functions give the rest of the notebook a compact, provider-neutral API. Notice that `embed()` can be built on top of `embed_many()`.

In [ ]:
def embed_many(*, model: EmbeddingModel, values: list[str]) -> EmbedManyResult:
    """Embed several values with a provider-neutral embedding model."""
    return model.do_embed(values)


def embed(*, model: EmbeddingModel, value: str) -> EmbedResult:
    """Embed one value using the same batch primitive."""
    result = embed_many(model=model, values=[value])
    return EmbedResult(
        value=result.values[0],
        embedding=result.embeddings[0],
        usage=result.usage,
        raw_response=result.raw_response,
    )


def generate_text(
    *,
    model: LanguageModel,
    instructions: str,
    prompt: str,
) -> GenerateTextResult:
    """Generate text with a provider-neutral language model."""
    return model.do_generate(instructions=instructions, prompt=prompt)

In [ ]:
provider = OpenAIProvider(
    api_key=os.environ["OPENAI_API_KEY"],
    base_url=openai_base_url,
)
embedding_model = provider.embedding_model(embedding_model_id)
language_model = provider.language_model(chat_model_id)

## 4. Embedding Similarity Primer

Before loading a PDF, let us use our own helpers to make dense vector retrieval concrete. An embedding model turns each string into a list of numbers. Related texts should land closer together in embedding space.

We will calculate cosine similarity ourselves:

```text
cosine_similarity(a, b) = dot_product(a, b) / (length(a) * length(b))
```

In [ ]:
def cosine_similarity(vector_a: list[float], vector_b: list[float]) -> float:
    """Measure the angle between two non-zero vectors."""
    if not vector_a or not vector_b:
        raise ValueError("Cosine similarity requires two non-empty vectors.")
    if len(vector_a) != len(vector_b):
        raise ValueError("Cosine similarity requires vectors with matching dimensions.")

    dot_product = sum(a * b for a, b in zip(vector_a, vector_b))
    length_a = sqrt(sum(a * a for a in vector_a))
    length_b = sqrt(sum(b * b for b in vector_b))

    if length_a == 0 or length_b == 0:
        raise ValueError("Cosine similarity is undefined for zero-length vectors.")

    return dot_product / (length_a * length_b)

In [ ]:
example_texts = [
    "king",
    "queen",
    "banana",
    "cat",
    "veterinarian",
    "cat health guidelines",
]

example_result = embed_many(model=embedding_model, values=example_texts)
example_vectors = dict(zip(example_result.values, example_result.embeddings))

comparison_pairs = [
    ("king", "queen"),
    ("king", "banana"),
    ("cat", "veterinarian"),
    ("cat", "cat health guidelines"),
]

for left, right in comparison_pairs:
    score = cosine_similarity(example_vectors[left], example_vectors[right])
    print(f"{left:>22} <> {right:<22} score={score:.3f}")

print("\nEmbedding API usage:", example_result.usage)

A few important notes:

- The score is useful for ranking, not as an absolute truth about meaning.
- Different embedding models can produce different scores.
- In RAG, we embed document chunks once, then embed each user query and search for nearby chunk vectors.

That is the retrieval part of RAG.

## 5. Documents - Loading the Cat Health Guideline PDF

Our internal `Document` type mirrors a common library abstraction:

- `page_content`: the text the retriever will search
- `metadata`: information such as source file and page number

`pypdf` extracts text from text-based PDFs. If the PDF contains scanned images, extraction may return little or no text and OCR would be needed.

In [ ]:
@dataclass
class Document:
    page_content: str
    metadata: dict[str, object]


def load_pdf(path: Path) -> list[Document]:
    """Load one Document per text-containing PDF page."""
    if not path.exists():
        raise FileNotFoundError(
            f"Expected the cat health guideline PDF at: {path.resolve()}\n"
            "The bundled course PDF is missing from this copy of the materials."
        )

    pages = []
    reader = PdfReader(str(path))

    for page_number, page in enumerate(reader.pages):
        text = (page.extract_text() or "").strip()
        if not text:
            continue

        pages.append(
            Document(
                page_content=text,
                metadata={
                    "source": path.name,
                    "page": page_number,
                    "document_type": "cat_health_guideline",
                },
            )
        )

    if not pages:
        raise ValueError(
            "The PDF loaded, but no extractable text was found. "
            "This usually means the PDF is scanned and needs OCR first."
        )

    return pages

In [ ]:
pdf_path = Path("data/cat_health_guidelines.pdf")
pages = load_pdf(pdf_path)

print(f"Loaded {len(pages)} text-containing PDF pages.")

In [ ]:
print(pages[0].page_content[:750])
print("\nMetadata:", pages[0].metadata)

## 6. Chunking the Documents

A page can be too large or too mixed-topic for focused retrieval. Libraries provide sophisticated splitters. Here we will use explicit overlapping character windows so the mechanics stay visible.

The overlap carries some local context from one chunk into the next. The `start_index` metadata lets us trace each chunk back to its source page.

In [ ]:
def split_documents(
    documents: list[Document],
    *,
    chunk_size: int,
    chunk_overlap: int,
) -> list[Document]:
    """Split documents into overlapping character windows."""
    if chunk_size <= 0:
        raise ValueError("chunk_size must be greater than zero.")
    if chunk_overlap < 0:
        raise ValueError("chunk_overlap must not be negative.")
    if chunk_overlap >= chunk_size:
        raise ValueError("chunk_overlap must be smaller than chunk_size.")

    step = chunk_size - chunk_overlap
    splits = []

    for document in documents:
        text = document.page_content

        for start_index in range(0, len(text), step):
            chunk_text = text[start_index : start_index + chunk_size].strip()
            if not chunk_text:
                continue

            splits.append(
                Document(
                    page_content=chunk_text,
                    metadata={
                        **document.metadata,
                        "chunk_id": f"chunk-{len(splits):04d}",
                        "start_index": start_index,
                    },
                )
            )

            if start_index + chunk_size >= len(text):
                break

    return splits

In [ ]:
chunk_size = 1000
chunk_overlap = 200

splits = split_documents(
    pages,
    chunk_size=chunk_size,
    chunk_overlap=chunk_overlap,
)

print(f"Split {len(pages)} pages into {len(splits)} chunks.")
print(f"Chunk size: {chunk_size} characters")
print(f"Chunk overlap: {chunk_overlap} characters")

In [ ]:
sample_chunk = splits[0]
print(sample_chunk.page_content[:750])
print("\nMetadata:", sample_chunk.metadata)

## 7. Build an In-Memory Vector Store

A vector database stores embeddings alongside documents and searches for nearby vectors efficiently. For a small teaching corpus, we can make that contract explicit with a Python list and brute-force cosine similarity.

The `add_documents()` method embeds chunks once during indexing. The `similarity_search_with_score()` method embeds a query at search time, compares it with every stored chunk vector, sorts the results, and returns the top `k` chunks.

In [ ]:
@dataclass
class StoredVector:
    document: Document
    embedding: list[float]


class InMemoryVectorStore:
    def __init__(self, embedding_model: EmbeddingModel):
        self.embedding_model = embedding_model
        self.records: list[StoredVector] = []

    def add_documents(self, documents: list[Document]) -> None:
        """Embed and store documents in memory."""
        if not documents:
            raise ValueError("At least one document is required for indexing.")

        result = embed_many(
            model=self.embedding_model,
            values=[document.page_content for document in documents],
        )

        self.records.extend(
            StoredVector(document=document, embedding=embedding)
            for document, embedding in zip(documents, result.embeddings)
        )

    def similarity_search_with_score(self, query: str, *, k: int) -> list[tuple[Document, float]]:
        """Return the top-k documents ranked by cosine similarity."""
        if not query.strip():
            raise ValueError("A non-empty retrieval query is required.")
        if k <= 0:
            raise ValueError("k must be greater than zero.")
        if not self.records:
            raise ValueError("Add documents before searching the vector store.")

        query_embedding = embed(model=self.embedding_model, value=query).embedding
        scored_documents = [
            (record.document, cosine_similarity(query_embedding, record.embedding))
            for record in self.records
        ]

        return sorted(scored_documents, key=lambda item: item[1], reverse=True)[:k]

In [ ]:
vector_store = InMemoryVectorStore(embedding_model=embedding_model)
vector_store.add_documents(splits)

print(f"Embedded chunks with: {embedding_model_id}")
print(f"Stored {len(vector_store.records)} chunk vectors in memory.")

## 8. Retrieval with Scores

Before generating an answer, inspect retrieval directly. If retrieval returns poor context, the final answer will usually be poor too.

The value `k` controls how many chunks the retriever returns. A larger `k` gives the language model more context, but it can also add noise.

In [ ]:
def display_retrieval_results(query: str, *, k: int) -> list[tuple[Document, float]]:
    """Retrieve chunks and print a compact view of the results."""
    results = vector_store.similarity_search_with_score(query, k=k)

    for index, (document, score) in enumerate(results, start=1):
        page = document.metadata.get("page")
        page_display = page + 1 if isinstance(page, int) else "unknown"
        chunk_id = document.metadata.get("chunk_id", "unknown")
        start_index = document.metadata.get("start_index", "unknown")
        preview = document.page_content[:350].replace("\n", " ")

        print(
            f"Source {index} | score={score:.3f} | page={page_display} | "
            f"chunk_id={chunk_id} | start_index={start_index}"
        )
        print(preview)
        print("-" * 100)

    return results

In [ ]:
retrieval_k = 4
retrieval_query = "What signs suggest that a cat should be seen by a veterinarian?"
retrieved_results = display_retrieval_results(retrieval_query, k=retrieval_k)

## 9. Retrieval Augmented Generation

Now we combine retrieval with generation:

1. Embed the question and retrieve relevant chunks.
2. Convert those chunks back into source-labeled natural-language context.
3. Insert the context into the prompt.
4. Generate a grounded answer.

This is still intentionally simpler than an agent. We always retrieve before answering.

In [ ]:
RAG_INSTRUCTIONS = """You are a cat health guideline assistant in a vector RAG lesson.

Use only the provided context to answer the user's question.
If the context does not contain enough information, say: "I don't have enough information in the provided cat health guideline PDF to answer that."

Cite the retrieved sources inline using labels like [Source 1] or [Source 2].
Do not diagnose, prescribe medication, or replace a veterinarian.
For diagnosis, treatment decisions, medication questions, or urgent symptoms, recommend contacting a veterinarian.
Keep the answer concise and practical."""


def format_context(scored_documents: list[tuple[Document, float]]) -> str:
    """Convert retrieved documents into a source-labeled context string."""
    formatted_chunks = []

    for index, (document, score) in enumerate(scored_documents, start=1):
        page = document.metadata.get("page")
        page_display = page + 1 if isinstance(page, int) else "unknown"
        source = document.metadata.get("source", "unknown source")

        formatted_chunks.append(
            f"[Source {index}] {source}, page {page_display}, score {score:.3f}\n"
            f"{document.page_content}"
        )

    return "\n\n".join(formatted_chunks)


def answer_question(question: str, *, k: int) -> dict[str, Any]:
    """Run retrieve-then-generate and return the answer plus source metadata."""
    scored_documents = vector_store.similarity_search_with_score(question, k=k)
    context = format_context(scored_documents)
    prompt = f"""Context:
{context}

Question: {question}

Answer from the context above."""
    generation = generate_text(
        model=language_model,
        instructions=RAG_INSTRUCTIONS,
        prompt=prompt,
    )

    sources = []
    for index, (document, score) in enumerate(scored_documents, start=1):
        page = document.metadata.get("page")
        sources.append(
            {
                "source_label": f"Source {index}",
                "file": document.metadata.get("source"),
                "page": page + 1 if isinstance(page, int) else None,
                "chunk_id": document.metadata.get("chunk_id"),
                "start_index": document.metadata.get("start_index"),
                "score": score,
            }
        )

    return {
        "question": question,
        "answer": generation.text,
        "sources": sources,
        "context": scored_documents,
        "generation_usage": generation.usage,
    }

Before calling the model, inspect the formatted context. This is the exact retrieved text that will be inserted into the RAG prompt.

In [ ]:
example_context = format_context(retrieved_results[:2])
print(example_context[:2000])

In [ ]:
answer_k = 4

result = answer_question(
    "What are signs that my cat may need veterinary attention?",
    k=answer_k,
)

print(result["answer"])
print("\nSources:")
for source in result["sources"]:
    print(source)
print("\nGeneration API usage:", result["generation_usage"])

### Additional Example Queries

Run a few questions that should be answerable from a cat health guideline PDF. The final unrelated question demonstrates the insufficient-context response.

In [ ]:
vibe_check_questions = [
    "What preventive care is recommended for cats?",
    "What symptoms should make me call a veterinarian?",
    "What should I know about feeding a healthy adult cat?",
    "Can my cat help me file my taxes?",
]

for question in vibe_check_questions:
    print("Question:", question)
    print(answer_question(question, k=answer_k)["answer"])
    print("=" * 100)

## Production Library Responsibilities

Our handcrafted pipeline is intentionally small and useful for learning. A production-ready library or vector database would usually take responsibility for more:

- Retries, rate-limit handling, timeouts, and exponential backoff
- Automatic batching, request-size limits, and concurrency controls
- Streaming, telemetry, tracing, and richer provider metadata
- Smarter text splitting and token-aware chunk sizes
- Persistent storage, indexes, metadata filtering, and scalable nearest-neighbor search
- Provider registries, middleware, caching, structured outputs, and tool calling

The important part is that these features extend the same core loop you implemented here.